# Sesión 08 — Lab 1: Pipeline Bronze → Silver con Expectations
## Lakeflow Spark Declarative Pipelines

**Curso:** Databricks Data Engineer Associate
**Runtime:** DBR 13.3 LTS o superior (recomendado: serverless)
**Plataforma:** Azure Databricks con Unity Catalog
**Dataset:** `ventas_diarias.csv`

---

### Objetivos del laboratorio
1. Crear un pipeline SDP/LDP con una capa Bronze (Streaming Table) y una capa Silver con expectations
2. Comprender la diferencia entre `@dp.expect`, `@dp.expect_or_drop` y `@dp.expect_or_fail`
3. Verificar el efecto de las expectations consultando Bronze y Silver

> **Importante:** Este notebook NO se ejecuta standalone. Es código fuente que se asocia a un Pipeline SDP. Funciones como `display()`, `dbutils.fs.ls()` o `spark.table().show()` no funcionan dentro de un pipeline. La verificación se hace en queries SQL aparte (último bloque de este notebook).

## Paso 1 — Subir el dataset al Volume

Antes de crear el pipeline, asegúrate de que el archivo CSV esté en la subcarpeta correcta del Volume:

```
/Volumes/dbassociate/default/vol_landing/sesion_08/ventas/ventas_diarias.csv
```

Si la carpeta `ventas/` no existe, créala desde el Catalog Explorer (Volumes → vol_landing → New folder → `sesion_08/ventas`).

## Paso 2 — Crear el Pipeline en la UI de Databricks

1. **Workspace → New → ETL Pipeline**
2. **Pipeline name:** `dbassociate_s08_lab1`
3. **Pipeline mode:** Triggered
4. **Source code:** este notebook (path completo)
5. **Destination:**
   - Catalog: `dbassociate`
   - Default schema: `bronze` (las tablas Silver usan full qualified name)
6. **Compute:** Serverless (recomendado)
7. **Channel:** Current
8. **Save** → **Start**

Después de crear el pipeline, las celdas de código de abajo serán las que SDP analice para construir el DAG.

In [0]:
# Importaciones del pipeline (API moderna)
# IMPORTANTE: NO usar 'import dlt' (legacy). La API actual es pyspark.pipelines.

from pyspark import pipelines as dp
from pyspark.sql import functions as F

## Paso 3 — Bronze: Streaming Table con Auto Loader

Lee los CSV desde el Volume con `STREAM read_files` (Auto Loader incremental). Cada archivo nuevo se procesa exactamente una vez. Se enriquece con metadata (`_ingested_at`, `_source_file`).

In [0]:
@dp.table(
    name="dbassociate.bronze.ventas",
    comment="Ingesta cruda de ventas diarias desde el Volume.",
    table_properties={"quality": "bronze"},
)
def bronze_ventas():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation",
                    "/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_ventas")
            .load("/Volumes/dbassociate/default/vol_landing/sesion_08/ventas/")
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
    )

## Paso 4 — Silver: Streaming Table con Expectations

Las expectations definen el contrato de calidad. Tres niveles:

| Decorador | Comportamiento |
|---|---|
| `@dp.expect("nombre", "cond")` | Warn (default) — registra en event log, NO descarta |
| `@dp.expect_or_drop("nombre", "cond")` | Descarta filas que no cumplen |
| `@dp.expect_or_fail("nombre", "cond")` | Falla el pipeline al primer registro inválido |

En este lab usamos `expect_or_drop` para datos inválidos pero no críticos, y `expect` para warnings.

In [0]:
@dp.table(
    name="dbassociate.silver.ventas",
    comment="Ventas validadas y tipadas. Filas con total <= 0 o fecha futura se descartan.",
    table_properties={"quality": "silver"},
)
@dp.expect_or_drop("total_positivo", "CAST(total AS DOUBLE) > 0")
@dp.expect_or_drop("fecha_valida", "to_date(fecha) <= current_date()")
@dp.expect("region_no_nula", "region IS NOT NULL")  # warn-only
def silver_ventas():
    return (
        spark.readStream.table("dbassociate.bronze.ventas")
            .select(
                F.col("venta_id"),
                F.to_date("fecha").alias("fecha"),
                F.col("sucursal"),
                F.col("producto_id"),
                F.col("cantidad").cast("int").alias("cantidad"),
                F.col("precio_unit").cast("decimal(10,2)").alias("precio_unit"),
                F.col("total").cast("decimal(12,2)").alias("total"),
                F.col("vendedor"),
                F.col("region"),
                F.col("_ingested_at"),
            )
    )

## Paso 5 — Ejecutar el pipeline

En la UI del pipeline (`dbassociate_s08_lab1`), pulsa **Start**. Espera a que el DAG muestre las dos tablas en verde.

Resultado esperado:
- `bronze.ventas`: 150 filas (todo el CSV)
- `silver.ventas`: ~145 filas (5 descartadas por expectations: 3 con total negativo o cero, 2 con fecha futura)

## Paso 6 — Verificación con SQL

Estas queries se corren en el SQL Editor o en un notebook normal (NO dentro del pipeline). Confirman el efecto de las expectations.

```sql
-- Conteo Bronze (sin filtros)
SELECT COUNT(*) AS bronze_count FROM dbassociate.bronze.ventas;

-- Conteo Silver (después de expectations)
SELECT COUNT(*) AS silver_count FROM dbassociate.silver.ventas;

-- Diferencia: las filas que las expectations descartaron
SELECT
    (SELECT COUNT(*) FROM dbassociate.bronze.ventas) -
    (SELECT COUNT(*) FROM dbassociate.silver.ventas) AS filas_descartadas;

-- Ver ejemplo de filas descartadas (existen en Bronze, no en Silver)
SELECT b.venta_id, b.fecha, b.total
FROM dbassociate.bronze.ventas b
LEFT ANTI JOIN dbassociate.silver.ventas s ON b.venta_id = s.venta_id;
```

## Antipatrones marcados

- `import dlt` → usar `from pyspark import pipelines as dp` (API moderna)
- `display(df)` o `dbutils.fs.ls()` dentro de este notebook → no se ejecuta interactivamente
- `spark.read.csv(...)` (batch) en lugar de `spark.readStream.format("cloudFiles")` → no es incremental
- Crear bronze como Materialized View → recomputaría todo cada vez
- Aplicar `@dp.expect_or_fail` para casos no críticos → aborta el pipeline ante cualquier fila mala
- Olvidar el `_schemas/` location de Auto Loader → falla la inferencia incremental

In [0]:
# LIMPIEZA — descomentar y ejecutar en SQL editor para restablecer el ambiente
# (NO dentro del pipeline; estas son operaciones imperativas)
#
# DROP TABLE IF EXISTS dbassociate.bronze.ventas;
# DROP TABLE IF EXISTS dbassociate.silver.ventas;
# -- También borrar el pipeline desde la UI
#
# Si quieres reiniciar Auto Loader, borra también el schema location:
# dbutils.fs.rm("/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_ventas", recurse=True)